# 🚀 AI LVQ Std BOT — Monthly Specialist Fine-Tuning (`kingescaped/lvq-std-BOT`)

Notebook huấn luyện tinh chỉnh (**Fine-tune**) hàng tháng cho 2 nhánh chuyên biệt trên repo Private Hugging Face **`kingescaped/lvq-std-BOT`**:
- **`btc-m15`**: Kéo tự động **1 năm (365 ngày)** nến M15 đã đóng của Bitcoin từ **Binance Public API** (`data-api.binance.vision`).
- **`xau-h1`**: Kéo tự động **2 năm (`730d`)** nến H1 đã đóng của Vàng từ **Yahoo Finance (`GC=F`)** *(hoặc tùy chọn đọc file CSV xuất từ MT5 Exness)*.

### Điều kiện trước khi bấm `Run all`:
1. Chọn Runtime GPU: **Runtime → Change runtime type → T4 GPU**.
2. Mở biểu tượng 🔑 **Secrets** ở thanh bên trái Colab, thêm khóa:
   - Tên: `HF_TOKEN`
   - Giá trị: Token quyền **Write** tạo tại `https://huggingface.co/settings/tokens` (bật công tắc *Notebook access*).

In [ ]:
# 1. Cài đặt thư viện & tải mã nguồn mới nhất từ GitHub
!pip install -q huggingface_hub safetensors einops tqdm yfinance pandas requests

import os
import subprocess

REPO_DIR = '/content/xauusd_signal_bot'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/kingescaped/xauusd_signal_bot.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)

In [ ]:
# 2. Đọc HF_TOKEN từ Colab Secrets & tự động khởi tạo/đồng bộ repo Private kingescaped/lvq-std-BOT
import os
import torch
from google.colab import userdata

HF_TOKEN = (userdata.get('HF_TOKEN') or '').strip()
os.environ['HF_TOKEN'] = HF_TOKEN
TARGET_REPO = 'kingescaped/lvq-std-BOT'

print('Thiết bị huấn luyện:', 'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU')

from hf_repo_sync import sync_base_to_private_repo
sync_info = sync_base_to_private_repo(target_repo=TARGET_REPO, token=HF_TOKEN)
print('Trạng thái repo Private:', sync_info)

In [ ]:
# 3. Fine-tune nhánh chuyên biệt BTC M15 (1 năm nến Binance Public API -> branch 'btc-m15')
from finetune_data import build_kronos_datasets, fetch_binance_btc_m15
from finetune_trainer import FineTuneConfig, run_specialist_finetune

btc_df = fetch_binance_btc_m15(days=365, symbol='BTCUSDT')
print(f'Đã tải {len(btc_df):,} nến BTC M15 đã đóng ({btc_df["time"].iloc[0]} -> {btc_df["time"].iloc[-1]})')

btc_train, btc_val = build_kronos_datasets(btc_df, context_len=200, pred_len=12, stride=4)
print(f'Tập mẫu BTC: train={len(btc_train[0]):,} cửa sổ | val={len(btc_val[0]):,} cửa sổ')

btc_cfg = FineTuneConfig(symbol='BTCUSD', branch='btc-m15', context_len=200, pred_len=12, epochs=5, batch_size=16, learning_rate=1e-6, weight_decay=0.1)
btc_metrics = run_specialist_finetune(btc_train, btc_val, btc_cfg, repo_id=TARGET_REPO, token=HF_TOKEN)
print('Kết quả BTC:', btc_metrics)

In [ ]:
# 4. Fine-tune nhánh chuyên biệt VÀNG H1 (Mặc định Yahoo Finance GC=F 730d -> branch 'xau-h1')
EXNESS_H1_CSV_PATH = ''  # Để trống để tự kéo Yahoo Finance GC=F (730d), hoặc điền đường dẫn CSV MT5 Exness nếu muốn

from finetune_data import fetch_gold_h1_candles

gold_df = fetch_gold_h1_candles(period='730d', ticker='GC=F', exness_csv_path=EXNESS_H1_CSV_PATH)
print(f'Đã tải {len(gold_df):,} nến Vàng H1 đã đóng ({gold_df["time"].iloc[0]} -> {gold_df["time"].iloc[-1]})')

gold_train, gold_val = build_kronos_datasets(gold_df, context_len=200, pred_len=6, stride=2)
print(f'Tập mẫu Vàng: train={len(gold_train[0]):,} cửa sổ | val={len(gold_val[0]):,} cửa sổ')

gold_cfg = FineTuneConfig(symbol='XAUUSD', branch='xau-h1', context_len=200, pred_len=6, epochs=5, batch_size=16, learning_rate=1e-6, weight_decay=0.1)
gold_metrics = run_specialist_finetune(gold_train, gold_val, gold_cfg, repo_id=TARGET_REPO, token=HF_TOKEN)
print('Kết quả Vàng:', gold_metrics)

In [ ]:
# 5. Kiểm tra nạp thử cả 2 nhánh từ repo Private kingescaped/lvq-std-BOT
import kronos_predictor

kronos_predictor._GLOBAL_PREDICTOR = None
pred_btc = kronos_predictor.load_kronos_base('BTCUSD', 'btc-m15')
pred_xau = kronos_predictor.load_kronos_base('XAUUSD', 'xau-h1')
print('Nguồn model BTC:', getattr(pred_btc, 'model_source', None))
print('Nguồn model XAU:', getattr(pred_xau, 'model_source', None))